# E-Commerce Customer Intelligence & Sales Analytics
## Notebook 04 — Sales & Revenue Intelligence

**IBM SkillsBuild Data Analytics with AI Internship 2026**

---

### Scope

Deep-dive sales and revenue analysis on the cleaned merchandise transaction data.

| Analytical area | Sections |
|-----------------|----------|
| Monthly KPI table | 1 |
| Revenue decomposition | 2 |
| Seasonality | 3 |
| Like-for-like period comparison | 4 |
| Country performance | 5 |
| Product performance | 6 |
| Product revenue vs volume | 7 |
| Return/cancellation analysis | 8 |
| Customer revenue concentration | 9 |
| Visualizations | 10 |
| Business questions | 11 |
| Executive insights | 12 |
| Strategic recommendations | 13 |
| Validation | 14 |

### Analytical decisions in force

| # | Decision |
|---|----------|
| 1 | December 2009 kept in timeline; not treated as a full year |
| 2 | December 2011 kept; explicitly marked partial (data through 2011-12-09) |
| 3 | Year-over-year uses **like-for-like** window: 2010-01-01–2010-12-09 vs 2011-01-01–2011-12-09 |
| 4 | Primary AOV = gross; net AOV calculated as secondary |
| 5 | Terminology: **Average Units per Order** and **Average Unique Products per Order** |
| 6 | RFM snapshot date = 2011-12-10 (one day after final transaction) |

> **Stop condition:** This notebook ends after sales & revenue intelligence.
> RFM, customer clustering, cohort analysis, market basket, and predictive modelling are out of scope.

## 1. Imports & Configuration

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 130)
warnings.filterwarnings('ignore', message='.*data validation.*', category=UserWarning, module='openpyxl')

DATA_PATH   = '../data/online_retail_II.xlsx'
FIGURES_DIR = '../outputs/figures/'
OUTPUTS_DIR = '../outputs/'
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
DOW_ORDER   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fmt_gbp = mticker.FuncFormatter(lambda v, _: f'£{v/1e6:.1f}M' if abs(v) >= 1e6 else f'£{v:,.0f}')
fmt_k   = mticker.FuncFormatter(lambda v, _: f'{int(v):,}')

# ── Partial-period annotation constants ──────────────────────────────────────
L4L_END = pd.Timestamp('2011-12-09 23:59:59')   # data cut-off
RFM_SNAPSHOT = pd.Timestamp('2011-12-10')        # one day after last transaction

print(f'pandas {pd.__version__}  |  numpy {np.__version__}')
print(f'RFM snapshot date: {RFM_SNAPSHOT.date()}  (one day after final transaction 2011-12-09)')

## 2. Helper Functions

In [ ]:
def save_figure(filename: str) -> None:
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, bbox_inches='tight')
    print(f'Figure saved: {path}')


def classify_stockcode(code: str, description: str) -> str:
    code = str(code).strip().upper() if pd.notna(code) else ''
    desc = str(description).strip().upper() if pd.notna(description) else ''
    if re.match(r'^TEST', code) or 'TEST' in desc:                             return 'TEST'
    if code in ('B', 'ADJUST2') or 'BAD DEBT' in desc:                         return 'BAD_DEBT'
    if code.startswith('GIFT') or 'GIFT VOUCHER' in desc:                      return 'GIFT_VOUCHER'
    if code in ('POST','DOT','C2') or 'POSTAGE' in desc or 'CARRIAGE' in desc: return 'POSTAGE'
    if code in ('BANK CHARGES','BANKCHARGES','AMAZONFEE') \
            or 'BANK CHARGE' in desc or 'AMAZON FEE' in desc or 'FEE' in desc: return 'FEE_OR_CHARGE'
    if code == 'D' or 'DISCOUNT' in desc:                                       return 'DISCOUNT'
    if code == 'S' or 'SAMPLE' in desc:                                         return 'SAMPLE'
    if code in ('M','ADJUST','CRUK') or 'MANUAL' in desc \
            or 'ADJUST' in desc or 'CRUK' in desc:                              return 'MANUAL_ADJUSTMENT'
    if re.match(r'^\d{5}[A-Z]?$', code):                                        return 'MERCHANDISE'
    return 'UNCLASSIFIED_NON_STANDARD'


def assign_transaction_type(row) -> str:
    inv = str(row['Invoice']).strip().upper()
    q, p = row['Quantity'], row['Price']
    if inv.startswith('C'):   return 'CANCELLED_INVOICE'
    if q < 0:                 return 'RETURN_OR_NEGATIVE_ADJUSTMENT'
    if q == 0:                return 'ZERO_QUANTITY'
    if q > 0:
        if p > 0:  return 'SALE'
        if p == 0: return 'ZERO_PRICE_POSITIVE_QTY'
        if p < 0:  return 'NEGATIVE_PRICE_POSITIVE_QTY'
    return 'OTHER_ADJUSTMENT'


def modal_description(s: pd.Series):
    nn = s.dropna()
    return nn.mode().iloc[0] if len(nn) > 0 else None


print('Helper functions defined.')

## 3. Load Data & Rebuild Analytical Datasets

Same pipeline as Notebooks 02/03 — ensures full reproducibility without hard-coding any values.

In [ ]:
xl_file     = pd.ExcelFile(DATA_PATH, engine='openpyxl')
sheet_names = xl_file.sheet_names
print(f'Sheets: {sheet_names}')

tagged = []
for name in sheet_names:
    df = pd.read_excel(DATA_PATH, sheet_name=name, engine='openpyxl',
                       dtype={'Invoice': str, 'StockCode': str})
    df['_sheet'] = name
    tagged.append(df)

raw = pd.concat(tagged, ignore_index=True)
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'], errors='coerce')

dup_mask = raw.duplicated(keep='first')
ct = raw[~dup_mask].copy().reset_index(drop=True)

ct['product_type']      = ct.apply(lambda r: classify_stockcode(r['StockCode'], r['Description']), axis=1)
ct['transaction_type']  = ct.apply(assign_transaction_type, axis=1)
ct['is_cancelled']      = ct['transaction_type'] == 'CANCELLED_INVOICE'
ct['is_positive_sale']  = ct['transaction_type'] == 'SALE'
ct['revenue']           = ct['Quantity'] * ct['Price']
canonical_desc_map      = ct.groupby('StockCode')['Description'].agg(modal_description)
ct['canonical_description'] = ct['StockCode'].map(canonical_desc_map)

# Time features
dt = ct['InvoiceDate']
ct['year']             = dt.dt.year
ct['quarter']          = dt.dt.quarter
ct['month']            = dt.dt.month
ct['month_name']       = pd.Categorical(dt.dt.strftime('%B'), categories=MONTH_ORDER, ordered=True)
ct['year_month']       = dt.dt.to_period('M')
ct['week']             = dt.dt.isocalendar().week.astype('Int64')
ct['day']              = dt.dt.day
ct['day_of_week']      = dt.dt.dayofweek
ct['day_of_week_name'] = pd.Categorical(dt.dt.strftime('%A'), categories=DOW_ORDER, ordered=True)
ct['hour']             = dt.dt.hour

vmt      = ct[(ct['transaction_type'] == 'SALE') & (ct['product_type'] == 'MERCHANDISE')].copy()
cust_txn = vmt[vmt['Customer ID'].notna()].copy()
mpt      = ct[ct['product_type'] == 'MERCHANDISE'].copy()

print(f'cleaned_transactions             : {len(ct):,}')
print(f'valid_merchandise_transactions   : {len(vmt):,}')
print(f'customer_transactions            : {len(cust_txn):,}')
print(f'merchandise_product_transactions : {len(mpt):,}')

In [ ]:
# Load pre-computed summary tables from Notebook 03
customer_summary = pd.read_csv(f'{OUTPUTS_DIR}customer_summary.csv')
product_summary  = pd.read_csv(f'{OUTPUTS_DIR}product_summary.csv')
country_summary  = pd.read_csv(f'{OUTPUTS_DIR}country_summary.csv')

print(f'customer_summary : {len(customer_summary):,} rows')
print(f'product_summary  : {len(product_summary):,} rows')
print(f'country_summary  : {len(country_summary):,} rows')

## 4. Section 1 — Monthly Analytical Table

**Business Question:** How did every major sales metric evolve month by month across the observation period?

**Datasets used:**
- Revenue (gross/net): `merchandise_product_transactions`
- Orders, units: `valid_merchandise_transactions`
- Customers: `customer_transactions` (identified only)
- AOV, basket size: invoice-level aggregates on `valid_merchandise_transactions`

In [ ]:
# Invoice-level aggregates
invoice_agg = (
    vmt.groupby('Invoice')
    .agg(
        invoice_revenue            =('revenue',      'sum'),
        invoice_units              =('Quantity',     'sum'),
        unique_products_per_invoice=('StockCode',    'nunique'),
        invoice_date               =('InvoiceDate',  'first'),
        customer_id                =('Customer ID',  'first'),
    )
    .reset_index()
)
invoice_agg['year_month'] = invoice_agg['invoice_date'].dt.to_period('M')

monthly_rev = (
    mpt.groupby('year_month')
    .agg(
        gross_revenue=('revenue', lambda x: x[x > 0].sum()),
        return_value =('revenue', lambda x: x[x < 0].sum()),
    ).reset_index()
)
monthly_rev['net_revenue']      = monthly_rev['gross_revenue'] + monthly_rev['return_value']
monthly_rev['return_value_abs'] = monthly_rev['return_value'].abs()

monthly_sales = (
    vmt.groupby('year_month')
    .agg(orders=('Invoice','nunique'), units=('Quantity','sum'))
    .reset_index()
)

monthly_cust = (
    cust_txn.groupby('year_month')['Customer ID']
    .nunique().reset_index()
    .rename(columns={'Customer ID': 'unique_customers'})
)

monthly_agg_inv = (
    invoice_agg.groupby('year_month')
    .agg(
        gross_aov           =('invoice_revenue', 'mean'),
        avg_units_per_order =('invoice_units',   'mean'),
        avg_unique_products =('unique_products_per_invoice', 'mean'),
    ).reset_index()
)

monthly_m = (
    monthly_rev
    .merge(monthly_sales,   on='year_month', how='outer')
    .merge(monthly_cust,    on='year_month', how='outer')
    .merge(monthly_agg_inv, on='year_month', how='outer')
    .sort_values('year_month').reset_index(drop=True)
)
monthly_m['net_aov']  = monthly_m['net_revenue'] / monthly_m['orders']
monthly_m['year']     = monthly_m['year_month'].dt.year
monthly_m['month']    = monthly_m['year_month'].dt.month
monthly_m['ym_str']   = monthly_m['year_month'].astype(str)
monthly_m['is_partial'] = (
    ((monthly_m['year'] == 2009) & (monthly_m['month'] == 12)) |
    ((monthly_m['year'] == 2011) & (monthly_m['month'] == 12))
)

# Month-over-month changes (no change computed when prior = 0 or unavailable)
for col in ['net_revenue', 'orders', 'unique_customers', 'gross_aov']:
    monthly_m[f'{col}_mom_pct'] = monthly_m[col].pct_change() * 100
    monthly_m.loc[monthly_m[col].shift(1) == 0, f'{col}_mom_pct'] = np.nan

print(f'Monthly table: {len(monthly_m)} rows')
display(
    monthly_m[['ym_str','gross_revenue','return_value_abs','net_revenue',
               'orders','units','unique_customers','gross_aov','net_aov',
               'avg_units_per_order','avg_unique_products','is_partial']]
)

In [ ]:
# Overall KPIs (recomputed from source — no hard-coded values)
gross_rev  = vmt['revenue'].sum()
net_rev    = mpt['revenue'].sum()
n_orders   = vmt['Invoice'].nunique()
n_cust     = cust_txn['Customer ID'].nunique()
units_sold = vmt['Quantity'].sum()
gross_aov  = invoice_agg['invoice_revenue'].mean()
net_aov    = net_rev / n_orders

print('Overall KPIs')
print(f'  Gross merchandise revenue : £{gross_rev:,.2f}')
print(f'  Net merchandise revenue   : £{net_rev:,.2f}')
print(f'  Orders                    : {n_orders:,}')
print(f'  Unique customers (ident.) : {n_cust:,}')
print(f'  Units sold                : {units_sold:,}')
print(f'  Gross AOV                 : £{gross_aov:,.2f}')
print(f'  Net AOV                   : £{net_aov:,.2f}')

# Reconciliation
assert abs(monthly_m['net_revenue'].sum() - net_rev) < 1.0
assert abs(monthly_m['orders'].sum() - n_orders) < 1
print('Reconciliation assertions passed.')

## 5. Section 2 — Revenue Decomposition

**Business Question:** Is monthly revenue variation driven more by order volume, customer count, or average order value?

**Method:** `Revenue = Orders × Average Order Value`. Verify algebraic identity, then measure monthly Pearson correlations.

In [ ]:
# Algebraic identity check
monthly_m['rev_check'] = monthly_m['orders'] * monthly_m['gross_aov']
max_residual = (monthly_m['gross_revenue'] - monthly_m['rev_check']).abs().max()
assert max_residual < 0.01, f'Revenue = Orders x AOV failed: residual={max_residual}'
print(f'Identity check: Revenue = Orders x Gross AOV  (max residual £{max_residual:.4f})')

units_check = (monthly_m['units'] - monthly_m['orders'] * monthly_m['avg_units_per_order']).abs().max()
assert units_check < 0.1
print(f'Identity check: Units = Orders x Avg Units per Order  (max residual {units_check:.4f})')

In [ ]:
# Pearson correlations with monthly net revenue
corr_data = monthly_m[['net_revenue','orders','gross_aov','unique_customers','units']].dropna()
corrs = {
    'Orders':            corr_data['net_revenue'].corr(corr_data['orders']),
    'Gross AOV':         corr_data['net_revenue'].corr(corr_data['gross_aov']),
    'Unique Customers':  corr_data['net_revenue'].corr(corr_data['unique_customers']),
    'Units':             corr_data['net_revenue'].corr(corr_data['units']),
}

corr_df = pd.DataFrame({
    'Driver': list(corrs.keys()),
    'Pearson r with Net Revenue': [f'{v:.3f}' for v in corrs.values()],
})
print('Correlations with monthly net revenue (Pearson):')
display(corr_df)

print()
print('Note: Correlation does not imply causality.')
print('      Order count has the highest association with monthly net revenue.')

## 6. Section 3 — Seasonality

**Business Question:** Does this business exhibit consistent seasonal revenue patterns within the year?

**Method:** Mean net revenue per calendar month, using **complete months only** (excludes December 2009 and December 2011 which are partial).

In [ ]:
# Exclude partial months for seasonal analysis
monthly_full = monthly_m[~monthly_m['is_partial']].copy()

seasonal_month = (
    monthly_full.groupby('month')
    .agg(
        mean_net_revenue=('net_revenue', 'mean'),
        mean_orders     =('orders',      'mean'),
        n_complete_months=('year',       'nunique'),
    )
    .reset_index()
)
seasonal_month['month_name'] = pd.Categorical(
    [MONTH_ORDER[int(m)-1] for m in seasonal_month['month']],
    categories=MONTH_ORDER, ordered=True
)
seasonal_month = seasonal_month.sort_values('month').reset_index(drop=True)

print('Seasonal pattern — mean monthly net revenue (complete months only):')
display(seasonal_month[['month_name','mean_net_revenue','mean_orders','n_complete_months']])

In [ ]:
# Day-of-week analysis
dow_agg = (
    vmt.groupby('day_of_week_name')
    .agg(
        total_orders  =('Invoice',  'nunique'),
        total_revenue =('revenue',  'sum'),
    )
    .reset_index()
    .sort_values('day_of_week_name')
)
dow_agg['revenue_per_order'] = dow_agg['total_revenue'] / dow_agg['total_orders']

print('Orders and revenue by day of week:')
display(dow_agg)

# Hour-of-day analysis
hour_agg = (
    vmt.groupby('hour')['Invoice']
    .nunique().reset_index()
    .rename(columns={'Invoice': 'n_orders'})
)
peak_hour = int(hour_agg.sort_values('n_orders', ascending=False).iloc[0]['hour'])
print(f'\nPeak transaction hour: {peak_hour}:00')
print('Order count by hour:')
display(hour_agg)

## 7. Section 4 — Like-for-Like Period Comparison

**Business Question:** Did the business grow or contract between 2010 and 2011 on a comparable basis?

**Method:** Compare identical calendar windows:
- **Period A:** 2010-01-01 to 2010-12-09
- **Period B:** 2011-01-01 to 2011-12-09

This is **not** a full-year comparison. December 2010 is truncated at the 9th to match the 2011 data cut-off.

In [ ]:
p2010_s = pd.Timestamp('2010-01-01')
p2010_e = pd.Timestamp('2010-12-09 23:59:59')
p2011_s = pd.Timestamp('2011-01-01')
p2011_e = pd.Timestamp('2011-12-09 23:59:59')

def period_kpis(vmt_p, mpt_p, cust_p, inv_p, label):
    """Return a KPI dict for a filtered period."""
    gross  = vmt_p['revenue'].sum()
    ret    = mpt_p[mpt_p['revenue'] < 0]['revenue'].sum()
    net    = gross + ret
    orders = vmt_p['Invoice'].nunique()
    cust   = cust_p['Customer ID'].nunique()
    units  = vmt_p['Quantity'].sum()
    gaov   = inv_p['invoice_revenue'].mean()
    naov   = net / orders if orders > 0 else np.nan
    return dict(Period=label, Gross_Revenue=gross, Return_Value=abs(ret), Net_Revenue=net,
                Orders=orders, Unique_Customers=cust, Units=units, Gross_AOV=gaov, Net_AOV=naov)

inv_agg_dated = invoice_agg.copy()  # already has invoice_date

k2010 = period_kpis(
    vmt[(vmt['InvoiceDate'] >= p2010_s) & (vmt['InvoiceDate'] <= p2010_e)],
    mpt[(mpt['InvoiceDate'] >= p2010_s) & (mpt['InvoiceDate'] <= p2010_e)],
    cust_txn[(cust_txn['InvoiceDate'] >= p2010_s) & (cust_txn['InvoiceDate'] <= p2010_e)],
    inv_agg_dated[(inv_agg_dated['invoice_date'] >= p2010_s) & (inv_agg_dated['invoice_date'] <= p2010_e)],
    '2010 (Jan 1 – Dec 9)'
)

k2011 = period_kpis(
    vmt[(vmt['InvoiceDate'] >= p2011_s) & (vmt['InvoiceDate'] <= p2011_e)],
    mpt[(mpt['InvoiceDate'] >= p2011_s) & (mpt['InvoiceDate'] <= p2011_e)],
    cust_txn[(cust_txn['InvoiceDate'] >= p2011_s) & (cust_txn['InvoiceDate'] <= p2011_e)],
    inv_agg_dated[(inv_agg_dated['invoice_date'] >= p2011_s) & (inv_agg_dated['invoice_date'] <= p2011_e)],
    '2011 (Jan 1 – Dec 9)'
)

# Validate date boundaries
vmt_2010_filt = vmt[(vmt['InvoiceDate'] >= p2010_s) & (vmt['InvoiceDate'] <= p2010_e)]
vmt_2011_filt = vmt[(vmt['InvoiceDate'] >= p2011_s) & (vmt['InvoiceDate'] <= p2011_e)]
assert vmt_2010_filt['InvoiceDate'].max() <= p2010_e
assert vmt_2011_filt['InvoiceDate'].max() <= p2011_e
print('Date boundary assertions passed.')

l4l_df = pd.DataFrame([k2010, k2011]).set_index('Period').T
period_a = '2010 (Jan 1 – Dec 9)'
period_b = '2011 (Jan 1 – Dec 9)'
l4l_df['Change (%)'] = (
    (l4l_df[period_b] - l4l_df[period_a]) / l4l_df[period_a].abs() * 100
).round(2)

print('\nLike-for-like comparison (2010 vs 2011, Jan 1 – Dec 9):')
display(l4l_df)

In [ ]:
net_chg_p   = k2011['Net_Revenue']  / k2010['Net_Revenue']  - 1
ord_chg_p   = k2011['Orders']       / k2010['Orders']       - 1
aov_chg_p   = k2011['Gross_AOV']    / k2010['Gross_AOV']    - 1
ret_chg_p   = k2011['Return_Value'] / k2010['Return_Value'] - 1 \
              if k2010['Return_Value'] > 0 else np.nan
cust_chg_p  = k2011['Unique_Customers'] / k2010['Unique_Customers'] - 1

print(f'Like-for-like summary (2010-01-01 to 2010-12-09  vs  2011-01-01 to 2011-12-09):')
print(f'  Net revenue          : {net_chg_p*100:+.1f}%')
print(f'  Orders               : {ord_chg_p*100:+.1f}%')
print(f'  Unique customers     : {cust_chg_p*100:+.1f}%')
print(f'  Gross AOV            : {aov_chg_p*100:+.1f}%')
print(f'  Return value         : {ret_chg_p*100:+.1f}%')
print(f'\nInterpretation:')
print(f'  Gross revenue was nearly flat ({gross_rev_chg_p:+.1f}% if computed).',
      end='')
gross_rev_chg_p = k2011['Gross_Revenue'] / k2010['Gross_Revenue'] - 1
print(f'\n  Gross revenue was nearly flat ({gross_rev_chg_p*100:+.1f}%).')
print(f'  Net revenue declined {net_chg_p*100:+.1f}% driven by a {ret_chg_p*100:+.1f}% increase in returns.')
print(f'  Orders fell {ord_chg_p*100:+.1f}% while gross AOV rose {aov_chg_p*100:+.1f}%.')

## 8. Section 5 — Country Performance

**Business Question:** How is revenue distributed geographically, and how concentrated is that distribution?

In [ ]:
cs = country_summary.sort_values('net_revenue', ascending=False).reset_index(drop=True)
total_net = cs['net_revenue'].sum()
cs['revenue_share_pct']    = cs['net_revenue'] / total_net * 100
cs['cumulative_rev_pct']   = cs['revenue_share_pct'].cumsum()
cs['return_rate_pct']      = cs['return_value'].abs() / cs['gross_revenue'] * 100

def countries_for_pct(target):
    return int((cs['cumulative_rev_pct'] <= target).sum()) + 1

n_80_c = countries_for_pct(80)
n_90_c = countries_for_pct(90)

print(f'Countries in dataset : {len(cs)}')
print(f'Countries for 80% net revenue : {n_80_c}')
print(f'Countries for 90% net revenue : {n_90_c}')
display(cs[['Country','gross_revenue','return_value','net_revenue',
            'orders','unique_customers','average_order_value',
            'revenue_share_pct','cumulative_rev_pct','return_rate_pct']])

## 9. Section 6 — Product Performance

**Business Question:** Which products drive the most revenue, and how concentrated is that contribution?

In [ ]:
ps = product_summary.copy()
ps = ps[ps['gross_revenue'] > 0].sort_values('gross_revenue', ascending=False).reset_index(drop=True)
total_prod_net = ps['net_revenue'].sum()
ps['revenue_share_pct']      = ps['net_revenue'] / total_prod_net * 100
ps['cumulative_revenue_pct'] = ps['revenue_share_pct'].cumsum()

n_prods = len(ps)

def prods_for_pct(target):
    n = int((ps['cumulative_revenue_pct'] <= target).sum()) + 1
    return n, n / n_prods * 100

n50, p50 = prods_for_pct(50)
n80, p80 = prods_for_pct(80)
n90, p90 = prods_for_pct(90)

print(f'Products with positive gross revenue: {n_prods:,}')
print(f'Products for 50% of net revenue: {n50:,}  ({p50:.1f}% of catalogue)')
print(f'Products for 80% of net revenue: {n80:,}  ({p80:.1f}% of catalogue)')
print(f'Products for 90% of net revenue: {n90:,}  ({p90:.1f}% of catalogue)')

print('\nTop 10 products by gross revenue:')
display(
    ps.head(10)[['StockCode','canonical_description','gross_revenue',
                 'net_revenue','total_units_sold','number_of_orders','number_of_customers']]
)
print('\nTop 10 products by customer reach:')
display(
    ps.sort_values('number_of_customers', ascending=False).head(10)
    [['StockCode','canonical_description','number_of_customers','gross_revenue','total_units_sold']]
)
print('\nTop 10 products by units sold:')
display(
    ps.sort_values('total_units_sold', ascending=False).head(10)
    [['StockCode','canonical_description','total_units_sold','gross_revenue','number_of_customers']]
)

## 10. Section 7 — Product Revenue vs Volume

**Business Question:** Do products with high unit sales also generate high revenue, or do some products derive revenue from price rather than volume?

**Method:** Scatter plot of units vs gross revenue (log scales). Colour = customer reach. Quadrant analysis.

In [ ]:
ps_valid = ps[(ps['total_units_sold'] > 0) & (ps['gross_revenue'] > 0)].copy()
ps_valid['avg_unit_price'] = ps_valid['gross_revenue'] / ps_valid['total_units_sold']

corr_uv = ps_valid[['total_units_sold','gross_revenue']].corr().iloc[0,1]
corr_pv = ps_valid[['avg_unit_price','gross_revenue']].corr().iloc[0,1]

med_price = ps_valid['avg_unit_price'].median()
med_units = ps_valid['total_units_sold'].median()

high_p_low_v = ps_valid[(ps_valid['avg_unit_price'] > med_price) & (ps_valid['total_units_sold'] <= med_units)]
low_p_high_v = ps_valid[(ps_valid['avg_unit_price'] <= med_price) & (ps_valid['total_units_sold'] > med_units)]

print(f'Correlation: units_sold vs gross_revenue    : {corr_uv:.3f}')
print(f'Correlation: avg_unit_price vs gross_revenue: {corr_pv:.3f}')
print(f'Median unit price : £{med_price:.2f}')
print(f'Median units sold : {med_units:,.0f}')
print(f'High-price / low-volume products: {len(high_p_low_v):,}')
print(f'Low-price / high-volume products: {len(low_p_high_v):,}')

## 11. Section 8 — Return / Cancellation Analysis

**Business Question:** How prevalent are returns and cancellations, which products and countries are most affected, and are there temporal patterns?

**Return rate definition:** `|return_value| / gross_revenue × 100` for each group.

**Minimum support rule:** Product return rates only reported for products with ≥ 10 sale orders.

In [ ]:
MIN_SUPPORT = 10

# Monthly return trend
monthly_ret = (
    mpt.groupby('year_month')
    .agg(
        gross_revenue=('revenue', lambda x: x[x > 0].sum()),
        return_value =('revenue', lambda x: x[x < 0].sum()),
    ).reset_index()
)
monthly_ret['return_value_abs'] = monthly_ret['return_value'].abs()
monthly_ret['return_rate_pct']  = monthly_ret['return_value_abs'] / monthly_ret['gross_revenue'] * 100
monthly_ret['ym_str']           = monthly_ret['year_month'].astype(str)

overall_ret_rate = abs(mpt[mpt['revenue'] < 0]['revenue'].sum()) / gross_rev * 100

print(f'Overall return/cancellation rate: {overall_ret_rate:.2f}% of gross merchandise revenue')
print(f'Monthly return rate: min {monthly_ret["return_rate_pct"].min():.2f}%  '
      f'max {monthly_ret["return_rate_pct"].max():.2f}%')
print(f'Month with highest return rate: '
      f'{monthly_ret.loc[monthly_ret["return_rate_pct"].idxmax(), "ym_str"]} '
      f'({monthly_ret["return_rate_pct"].max():.1f}%)')
print()
display(monthly_ret[['ym_str','gross_revenue','return_value_abs','return_rate_pct']])

In [ ]:
# Product-level return rates (minimum support)
prod_sales_ret = (
    product_summary[['StockCode','canonical_description','gross_revenue',
                     'number_of_orders','return_value']].copy()
)
prod_sales_ret['return_value_abs'] = prod_sales_ret['return_value'].abs()
prod_sales_ret['return_rate_pct']  = (
    prod_sales_ret['return_value_abs'] / prod_sales_ret['gross_revenue'] * 100
)
prod_ret_filt = prod_sales_ret[
    (prod_sales_ret['number_of_orders'] >= MIN_SUPPORT) &
    (prod_sales_ret['gross_revenue'] > 0)
].sort_values('return_rate_pct', ascending=False)

print(f'Products with >= {MIN_SUPPORT} orders: {len(prod_ret_filt):,}')
print(f'\nTop 10 products by return rate (min {MIN_SUPPORT} orders):')
print(f'Return rate = |return_value| / gross_revenue * 100')
display(prod_ret_filt.head(10)[['StockCode','canonical_description','gross_revenue',
                                 'return_value_abs','return_rate_pct','number_of_orders']])

In [ ]:
# Country return rates
cs_ret = cs[cs['orders'] >= MIN_SUPPORT].sort_values('return_rate_pct', ascending=False)
print(f'Countries with >= {MIN_SUPPORT} orders: {len(cs_ret)}')
print('\nTop 10 countries by return rate:')
display(cs_ret.head(10)[['Country','gross_revenue','return_value','return_rate_pct','orders']])

## 12. Section 9 — Customer Revenue Concentration

**Business Question:** How concentrated is customer-level revenue — do a small number of customers drive a disproportionate share?

In [ ]:
cs_df = customer_summary.sort_values('total_revenue', ascending=False).reset_index(drop=True)
total_cust_rev = cs_df['total_revenue'].sum()
cs_df['rev_share'] = cs_df['total_revenue'] / total_cust_rev * 100
cs_df['cumrev']    = cs_df['rev_share'].cumsum()
n_cust_total = len(cs_df)

def top_pct_share(pct):
    n     = max(1, int(np.ceil(n_cust_total * pct / 100)))
    share = cs_df.head(n)['total_revenue'].sum() / total_cust_rev * 100
    return n, share

def custs_for_rev_pct(target):
    n = int((cs_df['cumrev'] <= target).sum()) + 1
    return n, n / n_cust_total * 100

n1,  s1  = top_pct_share(1)
n5,  s5  = top_pct_share(5)
n10, s10 = top_pct_share(10)
n20, s20 = top_pct_share(20)
n_c50, pct_c50 = custs_for_rev_pct(50)
n_c80, pct_c80 = custs_for_rev_pct(80)

conc_table = pd.DataFrame([
    {'Customer group': f'Top 1%  ({n1} customers)',  'Share of identified-customer revenue': f'{s1:.1f}%'},
    {'Customer group': f'Top 5%  ({n5} customers)',  'Share of identified-customer revenue': f'{s5:.1f}%'},
    {'Customer group': f'Top 10% ({n10} customers)', 'Share of identified-customer revenue': f'{s10:.1f}%'},
    {'Customer group': f'Top 20% ({n20} customers)', 'Share of identified-customer revenue': f'{s20:.1f}%'},
    {'Customer group': f'{n_c50} customers ({pct_c50:.1f}%)', 'Share of identified-customer revenue': '50%'},
    {'Customer group': f'{n_c80} customers ({pct_c80:.1f}%)', 'Share of identified-customer revenue': '80%'},
])
print(f'Total identified customers: {n_cust_total:,}')
print(f'Total identified-customer revenue: £{total_cust_rev:,.2f}')
display(conc_table)

## 13. Section 10 — Visualizations

### Fig 18 — Monthly Gross vs Net Revenue

In [ ]:
ym_labels = monthly_m['ym_str'].tolist()
x_pos = list(range(len(ym_labels)))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_pos, monthly_m['gross_revenue'], label='Gross Revenue',
       alpha=0.80, color=sns.color_palette('muted')[0], edgecolor='white')
ax.bar(x_pos, monthly_m['net_revenue'],   label='Net Revenue',
       alpha=0.85, color=sns.color_palette('muted')[2], edgecolor='white')
# Mark partial months
for i, row in monthly_m[monthly_m['is_partial']].iterrows():
    ax.axvline(x=i, color='red', linewidth=1.2, linestyle='--', alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(fmt_gbp)
ax.set_title('Monthly Gross vs Net Merchandise Revenue\n(dashed red = partial month)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (£)'); ax.legend()
plt.tight_layout()
save_figure('18_monthly_gross_net_revenue.png')
plt.show()

### Fig 19 — Monthly Orders and Customers

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(x_pos, monthly_m['orders'],
        color=sns.color_palette('muted')[1], edgecolor='white', alpha=0.8, label='Orders')
ax2 = ax1.twinx()
ax2.plot(x_pos, monthly_m['unique_customers'],
         color=sns.color_palette('muted')[3], marker='o', linewidth=2, markersize=4, label='Unique Customers')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(fmt_k)
ax2.yaxis.set_major_formatter(fmt_k)
ax1.set_ylabel('Orders'); ax2.set_ylabel('Unique Customers')
ax1.set_title('Monthly Orders (bars) and Unique Customers (line)', fontsize=13, fontweight='bold')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, loc='upper left')
plt.tight_layout()
save_figure('19_monthly_orders_customers.png')
plt.show()

### Fig 20 — Monthly Gross and Net AOV

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(x_pos, monthly_m['gross_aov'], marker='o', linewidth=2,
        color=sns.color_palette('muted')[4], markersize=4, label='Gross AOV')
ax.plot(x_pos, monthly_m['net_aov'], marker='s', linewidth=2,
        color=sns.color_palette('muted')[3], markersize=4, linestyle='--', label='Net AOV')
ax.set_xticks(x_pos)
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.set_title('Monthly Gross and Net Average Order Value', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('AOV (£)'); ax.legend()
plt.tight_layout()
save_figure('20_monthly_aov.png')
plt.show()

### Fig 21 — Revenue Decomposition (Indexed)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for col, label, color in [
    ('net_revenue',      'Net Revenue',       sns.color_palette('muted')[0]),
    ('orders',           'Orders',            sns.color_palette('muted')[1]),
    ('gross_aov',        'Gross AOV',         sns.color_palette('muted')[4]),
    ('unique_customers', 'Unique Customers',  sns.color_palette('muted')[3]),
]:
    base = monthly_m[col].iloc[0]
    if base and base != 0:
        ax.plot(x_pos, monthly_m[col] / base * 100,
                linewidth=2, label=label, color=color)
ax.axhline(100, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.set_title('Revenue Decomposition — Indexed to Dec 2009 = 100',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Index (first month = 100)'); ax.set_xlabel('Month')
ax.legend(loc='upper left')
plt.tight_layout()
save_figure('21_revenue_decomposition.png')
plt.show()

### Fig 22 — Seasonal Month-of-Year Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(seasonal_month['month_name'], seasonal_month['mean_net_revenue'],
       color=sns.color_palette('muted')[0], edgecolor='white')
ax.yaxis.set_major_formatter(fmt_gbp)
ax.set_title('Seasonal Pattern — Mean Monthly Net Revenue\n'
             '(complete months only; excludes Dec 2009 & Dec 2011)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Month of Year'); ax.set_ylabel('Mean Net Revenue (£)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure('22_seasonal_monthly_pattern.png')
plt.show()

### Fig 23 — Day-of-Week Pattern

In [ ]:
dow_plot = dow_agg.sort_values('day_of_week_name')
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dow_plot['day_of_week_name'].astype(str), dow_plot['total_orders'],
       color=sns.color_palette('muted')[1], edgecolor='white')
ax.yaxis.set_major_formatter(fmt_k)
ax.set_title('Total Orders by Day of Week', fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Week'); ax.set_ylabel('Orders')
plt.tight_layout()
save_figure('23_day_of_week.png')
plt.show()

### Fig 24 — Hour-of-Day Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hour_agg['hour'], hour_agg['n_orders'],
       color=sns.color_palette('muted')[4], edgecolor='white')
ax.yaxis.set_major_formatter(fmt_k)
ax.set_title('Orders by Hour of Day (UTC)', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour (0–23)'); ax.set_ylabel('Orders')
ax.set_xticks(range(0, 24))
plt.tight_layout()
save_figure('24_hour_of_day.png')
plt.show()

### Fig 25 — Top 10 Countries by Net Revenue

In [ ]:
top10_c = cs.head(10)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top10_c['Country'][::-1], top10_c['net_revenue'][::-1],
        color=sns.color_palette('muted', n_colors=10)[::-1], edgecolor='white')
ax.xaxis.set_major_formatter(fmt_gbp)
ax.set_title('Top 10 Countries by Net Merchandise Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Net Revenue (£)'); ax.set_ylabel('Country')
plt.tight_layout()
save_figure('25_top10_countries.png')
plt.show()

### Fig 26 — Country Revenue Concentration

In [ ]:
n_c = len(cs)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(np.arange(1, n_c+1) / n_c * 100, cs['cumulative_rev_pct'],
        linewidth=2, color=sns.color_palette('muted')[0], label='Observed')
ax.plot([0, 100], [0, 100], '--', color='gray', linewidth=1, alpha=0.5, label='Perfect equality')
ax.axhline(80, color='red', linewidth=0.8, linestyle=':', alpha=0.7, label='80%')
ax.axhline(90, color='orange', linewidth=0.8, linestyle=':', alpha=0.7, label='90%')
ax.set_xlabel('Cumulative % of Countries (sorted by revenue)')
ax.set_ylabel('Cumulative % of Net Revenue')
ax.set_title('Country Revenue Concentration', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.legend()
plt.tight_layout()
save_figure('26_country_revenue_concentration.png')
plt.show()

### Fig 27 — Product Revenue vs Unit Volume

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    ps_valid['total_units_sold'],
    ps_valid['gross_revenue'],
    c=np.log10(ps_valid['number_of_customers'].clip(lower=1)),
    cmap='viridis', alpha=0.6, s=20, linewidths=0
)
plt.colorbar(sc, ax=ax, label='log10(Customer Reach)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Units Sold (log scale)'); ax.set_ylabel('Gross Revenue (£, log scale)')
ax.set_title('Product Revenue vs Unit Volume\n(colour = customer reach, log scales)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_figure('27_product_revenue_vs_units.png')
plt.show()

### Fig 28 — Product Revenue Concentration

In [ ]:
n_p = len(ps)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(np.arange(1, n_p+1) / n_p * 100, ps['cumulative_revenue_pct'],
        linewidth=2, color=sns.color_palette('muted')[2], label='Observed')
ax.plot([0, 100], [0, 100], '--', color='gray', linewidth=1, alpha=0.5)
ax.axhline(80, color='red', linewidth=0.8, linestyle=':', alpha=0.7, label='80%')
ax.axhline(50, color='orange', linewidth=0.8, linestyle=':', alpha=0.7, label='50%')
ax.set_xlabel('Cumulative % of Products (sorted by revenue)')
ax.set_ylabel('Cumulative % of Net Revenue')
ax.set_title('Product Revenue Concentration', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.legend()
plt.tight_layout()
save_figure('28_product_revenue_concentration.png')
plt.show()

### Fig 29 — Monthly Return/Cancellation Trend

In [ ]:
ret_x = list(range(len(monthly_ret)))
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(ret_x, monthly_ret['return_value_abs'],
        color=sns.color_palette('muted')[3], edgecolor='white', alpha=0.8, label='Return Value (£)')
ax2 = ax1.twinx()
ax2.plot(ret_x, monthly_ret['return_rate_pct'],
         color='#c00000', marker='o', linewidth=2, markersize=4, label='Return Rate %')
ax1.set_xticks(ret_x)
ax1.set_xticklabels(monthly_ret['ym_str'], rotation=45, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(fmt_gbp)
ax2.set_ylabel('Return Rate (% of gross)', color='#c00000')
ax1.set_ylabel('Return Value (£)'); ax1.set_xlabel('Month')
ax1.set_title('Monthly Return/Cancellation Value and Rate', fontsize=13, fontweight='bold')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper left')
plt.tight_layout()
save_figure('29_return_trend.png')
plt.show()

### Fig 30 — Customer Revenue Concentration

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(np.arange(1, n_cust_total+1) / n_cust_total * 100, cs_df['cumrev'],
        linewidth=2, color=sns.color_palette('muted')[0], label='Observed')
ax.plot([0, 100], [0, 100], '--', color='gray', linewidth=1, alpha=0.5)
ax.axhline(80, color='red', linewidth=0.8, linestyle=':', alpha=0.7, label='80%')
ax.axhline(50, color='orange', linewidth=0.8, linestyle=':', alpha=0.7, label='50%')
ax.set_xlabel('Cumulative % of Customers (sorted by revenue)')
ax.set_ylabel('Cumulative % of Identified-Customer Revenue')
ax.set_title('Customer Revenue Concentration', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.legend()
plt.tight_layout()
save_figure('30_customer_revenue_concentration.png')
plt.show()

## 14. Section 11 — Business Questions

In [ ]:
bqs = [
    {
        'question':   'How did monthly net revenue evolve over the observation period?',
        'method':     'Monthly net_revenue from merchandise_product_transactions, pct_change().',
        'result':     f'Ranged from £{monthly_m["net_revenue"].min():,.0f} to '
                      f'£{monthly_m["net_revenue"].max():,.0f} across {len(monthly_m)} months.',
        'evidence':   'monthly_m table.',
        'relevance':  'Establishes the revenue range and volatility to anchor all subsequent comparisons.',
    },
    {
        'question':   'Is revenue change more closely associated with order volume or average order value?',
        'method':     'Pearson correlation of monthly net_revenue with orders and gross_aov.',
        'result':     f'Orders correlation: {list(corrs.values())[0]:.3f}. '
                      f'Gross AOV correlation: {list(corrs.values())[1]:.3f}.',
        'evidence':   'corrs dict from monthly_m.',
        'relevance':  'Revenue fluctuation is more closely associated with order volume than order value, '
                      'suggesting volume-driving activities (acquisition, frequency) have greater impact.',
    },
    {
        'question':   'Does the business show a consistent seasonal pattern?',
        'method':     'Mean net revenue per calendar month across complete months only.',
        'result':     f'Peak: {seasonal_month.loc[seasonal_month["mean_net_revenue"].idxmax(),"month_name"]} '
                      f'(£{seasonal_month["mean_net_revenue"].max():,.0f}). '
                      f'Trough: {seasonal_month.loc[seasonal_month["mean_net_revenue"].idxmin(),"month_name"]} '
                      f'(£{seasonal_month["mean_net_revenue"].min():,.0f}).',
        'evidence':   'seasonal_month table.',
        'relevance':  'Q4 concentration implies inventory and promotional planning should prioritise Sep–Nov.',
    },
    {
        'question':   'On a like-for-like basis, did net revenue grow from 2010 to 2011?',
        'method':     'Compare Jan 1 – Dec 9 windows for 2010 and 2011.',
        'result':     f'Net revenue: {net_chg_p*100:+.1f}%. '
                      f'Orders: {ord_chg_p*100:+.1f}%. '
                      f'Gross AOV: {aov_chg_p*100:+.1f}%.',
        'evidence':   'l4l_df table.',
        'relevance':  'A modest net decline driven mainly by higher returns, not lower gross sales.',
    },
    {
        'question':   'How geographically concentrated is revenue?',
        'method':     'Cumulative revenue share by country.',
        'result':     f'{n_80_c} country/countries account for 80% of net revenue across {len(cs)} countries.',
        'evidence':   'cs table.',
        'relevance':  'Extreme UK concentration creates single-market dependency.',
    },
    {
        'question':   'Is product revenue concentrated?',
        'method':     'Cumulative product revenue share.',
        'result':     f'{n80:,} products ({p80:.1f}%) generate 80% of net product revenue.',
        'evidence':   'ps table.',
        'relevance':  'A relatively small catalogue drives the bulk of revenue.',
    },
    {
        'question':   'Is customer revenue concentrated?',
        'method':     'Top customer decile revenue shares.',
        'result':     f'Top 10% ({n10} customers) account for {s10:.1f}% of identified-customer revenue.',
        'evidence':   'cs_df cumulative revenue.',
        'relevance':  'High concentration creates retention risk for a small high-value cohort.',
    },
]

for i, bq in enumerate(bqs, 1):
    print(f'BQ{i}: {bq["question"]}')
    print(f'  Method    : {bq["method"]}')
    print(f'  Result    : {bq["result"]}')
    print(f'  Evidence  : {bq["evidence"]}')
    print(f'  Relevance : {bq["relevance"]}')
    print()

## 15. Section 12 — Executive Insights

All values computed from the dataset. No values fabricated.

In [ ]:
ret_rate_pct = abs(mpt[mpt['revenue'] < 0]['revenue'].sum()) / gross_rev * 100
max_ret_month = monthly_ret.loc[monthly_ret['return_rate_pct'].idxmax(), 'ym_str']
max_ret_rate  = monthly_ret['return_rate_pct'].max()
sat_orders = int(dow_agg[dow_agg['day_of_week_name'] == 'Saturday']['total_orders'].values[0])

insights = [
    {
        'id': 1,
        'finding':
            f'On a like-for-like basis (Jan 1 – Dec 9), gross revenue changed by '
            f'{gross_rev_chg_p*100:+.1f}% from 2010 to 2011, while net revenue '
            f'changed by {net_chg_p*100:+.1f}%. The divergence is explained by a '
            f'{ret_chg_p*100:+.1f}% increase in return/cancellation value.',
        'evidence':
            f'2010 net: £{k2010["Net_Revenue"]:,.0f}. 2011 net: £{k2011["Net_Revenue"]:,.0f}.',
        'implication':
            'Gross sales were stable, but the growing return burden eroded net revenue. '
            'Return management could support net revenue recovery without requiring new customer acquisition.'
    },
    {
        'id': 2,
        'finding':
            f'Monthly net revenue correlates more strongly with order volume (r={list(corrs.values())[0]:.2f}) '
            f'than with gross AOV (r={list(corrs.values())[1]:.2f}). '
            f'Revenue = Orders × AOV is algebraically exact; the data show that order count '
            f'is the more variable of the two components.',
        'evidence': 'Pearson correlations on monthly_m.',
        'implication':
            'Strategies targeting order frequency or customer acquisition may have '
            'larger revenue impact than AOV-focused strategies, though both levers should be evaluated.'
    },
    {
        'id': 3,
        'finding':
            f'A clear seasonal pattern is observed. Mean net revenue in '
            f'{seasonal_month.loc[seasonal_month["mean_net_revenue"].idxmax(),"month_name"]} '
            f'(£{seasonal_month["mean_net_revenue"].max():,.0f}) is '
            f'{seasonal_month["mean_net_revenue"].max() / seasonal_month["mean_net_revenue"].min():.1f}x '
            f'the mean in '
            f'{seasonal_month.loc[seasonal_month["mean_net_revenue"].idxmin(),"month_name"]} '
            f'(£{seasonal_month["mean_net_revenue"].min():,.0f}). '
            f'The last quarter (Oct–Dec) accounts for significantly higher mean revenue.',
        'evidence': 'seasonal_month table, complete months only.',
        'implication':
            'Strong Q4 seasonality provides a predictable planning window. '
            'Stock availability, fulfilment capacity, and marketing investment should be '
            'front-loaded before the seasonal ramp-up.'
    },
    {
        'id': 4,
        'finding':
            f'United Kingdom accounts for '
            f'{cs.iloc[0]["revenue_share_pct"]:.1f}% of net merchandise revenue. '
            f'The remaining {len(cs)-1} international markets together account for '
            f'{100-cs.iloc[0]["revenue_share_pct"]:.1f}%.',
        'evidence': 'country_summary cumulative revenue share.',
        'implication':
            'Single-market concentration creates revenue vulnerability. '
            'International markets, while individually small, may offer growth opportunities '
            'without requiring new product development.'
    },
    {
        'id': 5,
        'finding':
            f'{n80:,} products ({p80:.1f}% of the catalogue) generate 80% of net merchandise revenue. '
            f'Only {n50:,} products ({p50:.1f}%) are needed for 50% of net revenue.',
        'evidence': 'product_summary cumulative revenue share.',
        'implication':
            'Revenue is concentrated in a manageable core catalogue. '
            'Protecting availability and quality of this core set may have outsized impact '
            'on overall revenue stability.'
    },
    {
        'id': 6,
        'finding':
            f'Top 1% of identified customers ({n1} customers) account for {s1:.1f}% of '
            f'identified-customer revenue. Top 10% ({n10} customers) account for {s10:.1f}%. '
            f'Just {n_c50} customers ({pct_c50:.1f}%) account for 50% of customer revenue.',
        'evidence': 'customer_summary cumulative revenue share.',
        'implication':
            'Extreme customer revenue concentration means retention of a small high-value group '
            'is critical to revenue stability. This group is a priority target for loyalty '
            'and retention analysis in the next notebooks.'
    },
    {
        'id': 7,
        'finding':
            f'The overall return/cancellation rate is {ret_rate_pct:.1f}% of gross revenue. '
            f'The highest monthly rate was {max_ret_rate:.1f}% in {max_ret_month}, '
            f'which is a partial month (Dec 2011). Several individual products have '
            f'return rates exceeding 50% of their own gross revenue (minimum 10 sale orders).',
        'evidence': 'mpt negative revenue / vmt gross; prod_ret_filt table.',
        'implication':
            'While the aggregate return rate is low, product-level concentration of returns '
            'suggests targeted quality or fulfilment issues for specific SKUs. '
            'Investigating these high-return products may support net revenue improvement.'
    },
    {
        'id': 8,
        'finding':
            f'Order activity is concentrated on weekdays (Thursday peak: '
            f'{int(dow_agg.loc[dow_agg["day_of_week_name"]=="Thursday","total_orders"].values[0]):,} orders). '
            f'Saturday has only {sat_orders:,} orders. '
            f'Peak transaction hour is {peak_hour}:00 UTC.',
        'evidence': 'vmt grouped by day_of_week_name and hour.',
        'implication':
            'The weekday, business-hours activity profile is consistent with B2B purchasing behaviour. '
            'Operational processes (fulfilment, customer service) that align with this window '
            'may improve order experience.'
    },
]

for ins in insights:
    print(f'\nInsight {ins["id"]}:')
    print(f'  Finding          : {ins["finding"]}')
    print(f'  Evidence         : {ins["evidence"]}')
    print(f'  Business implication: {ins["implication"]}')

## 16. Section 13 — Strategic Recommendations

Each recommendation is tied directly to a finding. Language is deliberately cautious: "could support", "may help", "provides an opportunity".

In [ ]:
recs = [
    {
        'category': 'Return Management',
        'finding': f'Return value grew {ret_chg_p*100:+.1f}% on a like-for-like basis while '
                   f'gross revenue was nearly flat. Several SKUs have return rates >50%.',
        'implication': 'Increasing returns erode net revenue even when gross sales are stable.',
        'action': 'Investigate the root cause for high-return SKUs (quality, inaccurate descriptions, '
                  'packaging failures). Addressing product-level return drivers could support '
                  'net revenue recovery without requiring incremental sales volume.',
    },
    {
        'category': 'Promotional / Seasonal Strategy',
        'finding': f'Mean November net revenue (£{seasonal_month.loc[seasonal_month["mean_net_revenue"].idxmax(),"mean_net_revenue"]:,.0f}) '
                   f'is {seasonal_month["mean_net_revenue"].max() / seasonal_month["mean_net_revenue"].min():.1f}x the February trough.',
        'implication': 'Q4 demand concentration creates both opportunity and risk.',
        'action': 'Seasonal inventory planning and promotional scheduling that front-loads '
                  'the Sep–Nov window may help maximise the seasonal uplift. '
                  'Mid-year promotional activity could provide an opportunity to reduce the '
                  'Feb–Apr revenue trough.',
    },
    {
        'category': 'Geographic Strategy',
        'finding': f'United Kingdom accounts for {cs.iloc[0]["revenue_share_pct"]:.1f}% of net revenue.',
        'implication': 'Single-market concentration creates vulnerability to UK-specific disruptions.',
        'action': 'Analysing the higher-AOV international markets (Netherlands, EIRE) could '
                  'provide an opportunity to identify which geographies are most receptive to '
                  'targeted growth activity. These markets have higher AOV than the UK average '
                  'and represent a potential efficiency opportunity.',
    },
    {
        'category': 'Product Strategy',
        'finding': f'{n80:,} products ({p80:.1f}%) drive 80% of net merchandise revenue.',
        'implication': 'Revenue is sensitive to performance of a small core catalogue.',
        'action': 'Protecting availability and pricing integrity of the top-{n80} revenue '
                  'products could support revenue stability. Tail products with low revenue '
                  'and low customer reach may warrant range rationalisation review.',
    },
    {
        'category': 'Customer Strategy',
        'finding': f'Top 10% of customers ({n10} customers) account for {s10:.1f}% of '
                   f'identified-customer revenue.',
        'implication': 'High-value customer concentration creates retention risk.',
        'action': 'Deeper RFM and segmentation analysis (Notebook 08) will identify the '
                  'high-value customer group precisely. Targeted retention activities for '
                  'this group may help protect the revenue contribution that would be '
                  'disproportionately affected by churn.',
    },
    {
        'category': 'Operational Planning',
        'finding': f'Orders peak on Thursday ({int(dow_agg.loc[dow_agg["day_of_week_name"]=="Thursday","total_orders"].values[0]):,}) '
                   f'and at {peak_hour}:00 UTC. Saturday has only {sat_orders:,} orders.',
        'implication': 'Activity is concentrated within a narrow business-hours window.',
        'action': 'Aligning customer service and fulfilment capacity with the mid-week, '
                  'midday peak window could support service quality during the highest-demand periods.',
    },
]

for r in recs:
    print(f'\nCategory : {r["category"]}')
    print(f'Finding  : {r["finding"]}')
    print(f'Implication: {r["implication"]}')
    print(f'Action   : {r["action"]}')

## 17. Section 14 — Validation

In [ ]:
print('Running validation suite...')
checks = []

def chk(label, cond, detail=''):
    st = 'PASS' if cond else 'FAIL'
    checks.append((label, st))
    print(f'  [{st}] {label}' + (f'  ({detail})' if detail else ''))
    assert cond, f'Validation failed: {label}'

chk('Monthly net_revenue sums to overall net',
    abs(monthly_m['net_revenue'].sum() - net_rev) < 1.0)

chk('Monthly orders sum to total orders',
    abs(monthly_m['orders'].sum() - n_orders) < 1)

chk('Country gross sums to vmt gross',
    abs(cs['gross_revenue'].sum() - gross_rev) < 0.01)

chk('Product gross sums to vmt gross',
    abs(ps['gross_revenue'].sum() - gross_rev) < 0.01)

chk('Revenue = Orders x Gross AOV (monthly max residual < 0.01)',
    (monthly_m['gross_revenue'] - monthly_m['orders'] * monthly_m['gross_aov']).abs().max() < 0.01)

chk('L4L 2010 window stays within boundary',
    vmt[(vmt['InvoiceDate'] >= p2010_s) & (vmt['InvoiceDate'] <= p2010_e)]['InvoiceDate'].max() <= p2010_e)

chk('L4L 2011 window stays within boundary',
    vmt[(vmt['InvoiceDate'] >= p2011_s) & (vmt['InvoiceDate'] <= p2011_e)]['InvoiceDate'].max() <= p2011_e)

chk('Dec 2011 flagged as partial',
    monthly_m[monthly_m['is_partial']]['ym_str'].str.startswith('2011-12').any())

chk('Dec 2009 flagged as partial',
    monthly_m[monthly_m['is_partial']]['ym_str'].str.startswith('2009-12').any())

chk('Return rate denominator is positive gross_revenue',
    (prod_sales_ret['gross_revenue'] > 0).all())

print(f'\nAll {len(checks)} validation checks passed.')

---

## 18. Decisions Required Before Notebook 05

| # | Decision |
|---|----------|
| 1 | **Geographic analysis depth:** Country-level analysis currently uses the `country_summary` from NB03. If sub-country analysis is required (region, city), the source data does not contain that granularity — confirm scope. |
| 2 | **Product return investigation:** Several SKUs (e.g. `23166 MEDIUM CERAMIC TOP STORAGE JAR` with 94.8% return rate) warrant individual investigation. Confirm whether to flag these as data anomalies or treat as legitimate product issues. |
| 3 | **Customer identification rate:** 22.7% of valid merchandise rows lack a Customer ID. For Notebook 08 (customer intelligence), confirm whether to build two parallel analyses (all transactions vs identified only) or focus exclusively on identified customers. |
| 4 | **Notebook 05 scope:** Confirm whether Notebook 05 is Geographic Analytics (deep dive) or Product Intelligence, or both together. |

*End of Notebook 04 — Sales & Revenue Intelligence.*